# Geração da Base de Colaboradores por Unidade/Dia

Calcula `quantidade_colaboradores` simulando três perfis reais de dimensionamento:

- **Carga diária** = Σ (volume_atendimentos × tempo_medio_min) por empresa/dia
- **Colaboradores ideal** = carga_média / 480 min por empresa
- **Perfil da empresa** (fixo, sorteado com seed 42):
  - `adequado` (~30% das empresas): 1,6× o ideal → utilização média ~62% (folga de pessoal)
  - `alerta`   (~40% das empresas): 1,2× o ideal → utilização média ~83% (quadro saudável)
  - `critico`  (~30% das empresas): 0,85× o ideal → utilização média ~118% (sobrecarga)
- **Variação diária** de ±1 colaborador quando o volume do dia desvia >15% da média da empresa

In [ ]:
import pandas as pd
import numpy as np

vol = pd.read_csv('../Bases/VolumeDeAtendimentos.csv', sep=';')
tempo = pd.read_csv('../Bases/TempoMedioAtendimentos.csv', sep=';')

vol['data'] = pd.to_datetime(vol['data'])
tempo['data'] = pd.to_datetime(tempo['data'])

# Normaliza empresa para formato EMP01, EMP02... (TempoMedio usa EMP01, Volume usa EMP1)
def normaliza_empresa(s):
    num = ''.join(filter(str.isdigit, s))
    return f"EMP{int(num):02d}"

vol['empresa'] = vol['empresa'].apply(normaliza_empresa)
tempo['empresa'] = tempo['empresa'].apply(normaliza_empresa)

print("Volume shape:", vol.shape)
print("Tempo shape:", tempo.shape)
print("Empresas em Volume:", sorted(vol['empresa'].unique()))
print("Empresas em Tempo:", sorted(tempo['empresa'].unique()))

In [ ]:
# Junta volume com tempo médio por (data, empresa, produto)
merged = vol.merge(tempo, on=['data', 'empresa', 'produto'], how='inner')

# Minutos totais por produto/dia = volume * tempo_medio
merged['minutos'] = merged['volume_atendimentos'] * merged['tempo_medio_min']

# Soma minutos por (data, empresa) — carga total do dia
carga_dia = (
    merged.groupby(['data', 'empresa'])['minutos']
    .sum()
    .reset_index()
    .rename(columns={'minutos': 'minutos_totais'})
)

print(f"Linhas após merge: {merged.shape[0]}")
print(f"Registros em carga_dia: {carga_dia.shape[0]}")
print(f"\nEstatísticas de minutos totais por empresa/dia:")
carga_dia['minutos_totais'].describe().round(1)

In [ ]:
MINUTOS_DIA = 480  # jornada de trabalho por colaborador

# Colaboradores base = carga / 480, arredondado, mínimo 1
carga_dia['colaboradores_exato'] = carga_dia['minutos_totais'] / MINUTOS_DIA
carga_dia['colaboradores_base'] = np.maximum(1, carga_dia['colaboradores_exato'].round().astype(int))

print("Distribuição dos colaboradores base (sem variação):")
print(carga_dia['colaboradores_base'].value_counts().sort_index())
print(f"\nMédia de colaboradores/empresa/dia: {carga_dia['colaboradores_base'].mean():.2f}")
carga_dia[['minutos_totais', 'colaboradores_exato', 'colaboradores_base']].describe().round(2)

In [ ]:
np.random.seed(42)

empresas = carga_dia['empresa'].unique()
n = len(empresas)

# Sorteia perfil para cada empresa
# ~30% adequado (excesso de staff)  → utilizacao < 70%
# ~40% alerta   (quadro ideal)      → utilizacao 70-90%
# ~30% critico  (escassez de staff) → utilizacao > 90%
idx = np.random.permutation(n)
n_adequado = round(n * 0.30)
n_critico  = round(n * 0.30)

# dtype=object evita truncamento de strings com comprimentos diferentes
perfil_arr = np.array(['alerta'] * n, dtype=object)
perfil_arr[idx[:n_adequado]]                       = 'adequado'
perfil_arr[idx[n_adequado:n_adequado + n_critico]] = 'critico'
perfil = pd.Series(perfil_arr, index=empresas, name='perfil')

# Carga media diaria por empresa (em colaboradores ideais)
ideal_empresa = (
    carga_dia.groupby('empresa')['minutos_totais']
    .mean()
    .div(MINUTOS_DIA)
    .rename('ideal_medio')
)

# Multiplicador sobre o ideal por perfil
# adequado: 1.6x ideal → util ~62%  (folga de pessoal)
# alerta  : 1.2x ideal → util ~83%  (dentro da faixa saudavel)
# critico : 0.85x ideal → util ~118% (sobrecarga)
MULT = {'adequado': 1.6, 'alerta': 1.2, 'critico': 0.85}

colabs_fixo = (
    ideal_empresa
    .mul(perfil.map(MULT))
    .apply(lambda x: max(1, round(x)))
    .astype(int)
    .rename('colabs_fixo')
)

carga_dia = carga_dia.join(perfil,        on='empresa')
carga_dia = carga_dia.join(ideal_empresa, on='empresa')
carga_dia = carga_dia.join(colabs_fixo,   on='empresa')

# Variacao diaria ±1 quando o volume do dia desvia >15% da media da empresa
media_emp = carga_dia.groupby('empresa')['minutos_totais'].transform('mean')
ratio     = carga_dia['minutos_totais'] / media_emp
ajuste    = np.where(ratio >= 1.15, 1, np.where(ratio <= 0.85, -1, 0))

carga_dia['quantidade_colaboradores'] = np.maximum(1, carga_dia['colabs_fixo'] + ajuste)

# Diagnostico
carga_dia['util_pct'] = (
    carga_dia['minutos_totais'] / (carga_dia['quantidade_colaboradores'] * MINUTOS_DIA) * 100
).round(1)

print("Empresas por perfil:")
print(perfil.value_counts())
print("\nUtilizacao media por perfil (%):")
print(carga_dia.groupby('perfil')['util_pct'].describe().round(1))
carga_dia[['empresa', 'perfil', 'colabs_fixo', 'quantidade_colaboradores', 'util_pct']].head(20)

In [ ]:
# Base final: data, empresa, quantidade_colaboradores
colaboradores = carga_dia[['data', 'empresa', 'quantidade_colaboradores']].copy()
colaboradores['data'] = colaboradores['data'].dt.strftime('%Y-%m-%d')

print(colaboradores.shape)
print(colaboradores['quantidade_colaboradores'].describe().round(2))
colaboradores.head(20)

In [ ]:
colaboradores.to_csv('../Bases/ColaboradoresPorDia.csv', sep=';', index=False)
print('Salvo em ../Bases/ColaboradoresPorDia.csv')